In [2]:
%reload_ext autoreload
%autoreload 2

In [1]:
# %%
import asyncio
import os
import pandas as pd
import tiktoken
from openai import AsyncOpenAI  # Import the asynchronous client
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

# %%
def get_data(dataset_path: str, corpus_split: str = "train", queries_split: str = "train", relevant_docs_split: str = "test"):
    """Loads corpus, queries, and relevant documents from the dataset."""
    corpus_dataset = load_dataset(dataset_path, data_files="corpus.jsonl", split=corpus_split)
    queries_dataset = load_dataset(dataset_path, data_files="queries.jsonl", split=queries_split)
    relevant_docs_dataset = load_dataset(dataset_path, split=relevant_docs_split)

    corpus = {row["_id"]: row["text"] for row in corpus_dataset}
    queries = {row["_id"]: row["text"] for row in queries_dataset}
    
    relevant_docs_data = relevant_docs_dataset.to_pandas().groupby("query-id")["corpus-id"].apply(set).to_dict()
    relevant_docs_data = {str(k): {str(item) for item in v} for k, v in relevant_docs_data.items()}

    return corpus, queries, relevant_docs_data

# %%
# --- Configuration ---
MODEL_NAME = "text-embedding-3-small"
MAX_TOKENS = 8192

# --- Initialize Tokenizer ---
tokenizer = tiktoken.encoding_for_model(MODEL_NAME)

def truncate_text(text: str, max_tokens: int = MAX_TOKENS) -> str:
    """Truncates a text string to a maximum number of tokens."""
    tokens = tokenizer.encode(text)
    if len(tokens) > max_tokens:
        truncated_tokens = tokens[:max_tokens]
        return tokenizer.decode(truncated_tokens)
    return text

# --- NEW: Asynchronous and Concurrent Embedding Generation ---

async def _create_embedding_df_async(
    client: AsyncOpenAI, 
    data: dict, 
    model: str, 
    batch_size: int, 
    semaphore: asyncio.Semaphore,
    desc: str
) -> pd.DataFrame:
    """
    Helper function to generate embeddings concurrently using asyncio.Semaphore.
    """
    ids = list(data.keys())
    texts = list(data.values())
    
    # This dictionary will store results, keyed by batch index to maintain order
    results_dict = {}

    async def get_embeddings_for_batch(batch_index: int, batch_texts: list[str]):
        """Worker coroutine to process one batch of texts."""
        async with semaphore: # Wait for the semaphore to allow a new request
            truncated_batch = [truncate_text(text) for text in batch_texts]
            
            response = await client.embeddings.create(
                input=truncated_batch, 
                model=model
            )
            
            # Store results using the batch index
            embeddings = [res.embedding for res in response.data]
            tokens_used = response.usage.total_tokens
            results_dict[batch_index] = (embeddings, tokens_used)

    # Create a list of tasks for all batches
    tasks = []
    for i, j in enumerate(range(0, len(texts), batch_size)):
        batch = texts[j:j + batch_size]
        task = asyncio.create_task(get_embeddings_for_batch(i, batch))
        tasks.append(task)
        
    # Run tasks concurrently with a progress bar
    for future in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=desc):
        await future  # Wait for each task to complete

    # Process results in the correct order
    all_embeddings = []
    total_tokens_used = 0
    sorted_results = [results_dict[i] for i in sorted(results_dict.keys())]
    
    for embeddings_batch, tokens_in_batch in sorted_results:
        all_embeddings.extend(embeddings_batch)
        total_tokens_used += tokens_in_batch

    return pd.DataFrame({
        'id': ids,
        'embeddings': all_embeddings,
        'total_tokens': [total_tokens_used] * len(ids)
    })

async def gen_openai_emb_async(
    corpus: dict, 
    queries: dict, 
    model: str = MODEL_NAME, 
    batch_size: int = 100,
    concurrency_limit: int = 25 # Max concurrent requests
):
    """
    Generates embeddings for a corpus and queries concurrently.
    """
    print("Initializing async client and semaphore...")
    client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    semaphore = asyncio.Semaphore(concurrency_limit)
    
    print("Creating concurrent tasks for corpus and query embeddings...")
    
    # Create two main tasks: one for the corpus, one for the queries
    corpus_task = _create_embedding_df_async(client, corpus, model, batch_size, semaphore, "Generating Corpus Embeddings")
    query_task = _create_embedding_df_async(client, queries, model, batch_size, semaphore, "Generating Query Embeddings")

    # Run both tasks in parallel and wait for them to complete
    corpus_embeddings_df, query_embeddings_df = await asyncio.gather(
        corpus_task,
        query_task
    )

    return corpus_embeddings_df, query_embeddings_df

# %%
async def main():
    """Main function to run the data loading and embedding generation."""
    dataset_path = "nasa-impact/nasa-sde-IR-benchmark-sample-v1"
    corpus, queries, relevant_docs_data = get_data(dataset_path)
    
    # Sample a small subset for testing
    corpus_sample = {k: corpus[k] for k in list(corpus.keys())[:1000]}
    queries_sample = {k: queries[k] for k in list(queries.keys())[:1000]}

    corpus_df, queries_df = await gen_openai_emb_async(corpus_sample, queries_sample)
    
    return corpus_df, queries_df


In [2]:
corpus_df, queries_df = await main()

Initializing async client and semaphore...
Creating concurrent tasks for corpus and query embeddings...


Generating Corpus Embeddings:   0%|          | 0/10 [00:00<?, ?it/s]

Generating Query Embeddings:   0%|          | 0/10 [00:00<?, ?it/s]

In [4]:
queries_df

,id,embeddings,total_tokens
0,q0,"[-0.01019254419952631, 0.04990934208035469, 0....",16934
1,q1,"[0.007902461104094982, -0.04599219560623169, -...",16934
2,q2,"[0.0076387180015444756, -0.04086777940392494, ...",16934
3,q3,"[0.00973544828593731, -0.02543921209871769, 0....",16934
4,q4,"[0.009164327755570412, 0.04319829121232033, 0....",16934
...,...,...,...
995,q995,"[0.022687502205371857, 0.009754028171300888, 0...",16934
996,q996,"[-0.0015263254754245281, 0.025505144149065018,...",16934
997,q997,"[-0.026209531351923943, 0.0015290571609511971,...",16934
998,q998,"[0.024597294628620148, 0.01805090717971325, 0....",16934


In [37]:
from sentence_transformers import util
from custum_evals import (
    MultiGPUInformationRetrievalEvaluator,  # Import the custom evaluator
)
import pandas as pd
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import torch

# Define a dummy placeholder for the model card data attribute
class DummyModelCardData:
    def set_evaluation_metrics(self, *args, **kwargs):
        """This is a dummy method. It does nothing."""
        pass

# Define the complete, self-contained dummy model
class DummyModel:
    """
    A standalone placeholder model that mimics all necessary attributes 
    and methods for the InformationRetrievalEvaluator when using 
    pre-computed embeddings.
    """
    def __init__(self):
        self.similarity = util.cos_sim
        self.similarity_fn_name = 'cosine'
        self.model_card_data = DummyModelCardData()

    def start_multi_process_pool(self, *args, **kwargs):
        """Dummy method. Returns an empty dict."""
        return {}

    def stop_multi_process_pool(self, pool):
        """Dummy method. Does nothing."""
        pass


# Instantiate our dummy model
dummy_model = DummyModel()


# testting the embeddings
embedding_path = "/rhome/sawale/indus_traning/sentense_transformers/eval/openai_emb_cache/text-embedding-3-large/nasa-sde-IR-benchmark-sample-v1"
dataset_path = "nasa-impact/nasa-sde-IR-benchmark-sample-v2"
corpus_df = pd.read_parquet(os.path.join(embedding_path, "corpus_embeddings.parquet"))
queries_df = pd.read_parquet(os.path.join(embedding_path, "queries_embeddings.parquet"))



relevant_docs_dataset = load_dataset(dataset_path, split="test")
relevant_docs_data = relevant_docs_dataset.to_pandas().groupby("query-id")["corpus-id"].apply(set).to_dict()
relevant_docs_data = {str(k): {str(item) for item in v} for k, v in relevant_docs_data.items()}




# 1. Create a dictionary for query embeddings: {query_id: embedding}
query_embeddings_dict = pd.Series(
    queries_df.embeddings.values, 
    index=queries_df.id
).to_dict()

# 2. Create a dictionary for corpus embeddings: {corpus_id: embedding}
corpus_embeddings_dict = pd.Series(
    corpus_df.embeddings.values, 
    index=corpus_df.id
).to_dict()


# 3. Use your existing relevant_docs dictionary
# This should already be in the format: {query_id: {relevant_corpus_id_1, ...}}
relevant_docs_dict = relevant_docs_data



In [ ]:

# Instantiate the evaluator with your prepared dictionaries
evaluator = MultiGPUInformationRetrievalEvaluator(
    queries=query_embeddings_dict,         # Your query embeddings
    corpus=corpus_embeddings_dict,         # Your corpus embeddings
    relevant_docs=relevant_docs_dict,      # Ground truth
    name="nasa-evaluation",                # A name for the evaluation
    show_progress_bar=True,
    main_score_function=None          # Use cosine similarity
)

# Run the evaluation
# The 'model' argument is required but not used for encoding in this case.
results = evaluator(model=dummy_model, 
                    corpus_embeddings=torch.Tensor(corpus_df.embeddings.values.tolist()),
                    query_embeddings=torch.Tensor(queries_df.embeddings.values.tolist())
                    )

Corpus Chunks: 0it [00:00, ?it/s]


IndexError: list index out of range

In [ ]:
results

{'nasa-evaluation_cosine_accuracy@1': 0.43733333333333335,
 'nasa-evaluation_cosine_accuracy@3': 0.5333333333333333,
 'nasa-evaluation_cosine_accuracy@5': 0.5846666666666667,
 'nasa-evaluation_cosine_accuracy@10': 0.6446666666666667,
 'nasa-evaluation_cosine_precision@1': 0.43733333333333335,
 'nasa-evaluation_cosine_precision@3': 0.17777777777777778,
 'nasa-evaluation_cosine_precision@5': 0.11693333333333332,
 'nasa-evaluation_cosine_precision@10': 0.06446666666666666,
 'nasa-evaluation_cosine_recall@1': 0.43733333333333335,
 'nasa-evaluation_cosine_recall@3': 0.5333333333333333,
 'nasa-evaluation_cosine_recall@5': 0.5846666666666667,
 'nasa-evaluation_cosine_recall@10': 0.6446666666666667,
 'nasa-evaluation_cosine_ndcg@10': 0.5343137763592799,
 'nasa-evaluation_cosine_mrr@10': 0.49987857142857145,
 'nasa-evaluation_cosine_map@100': 0.5068317168956665}

In [60]:
c = load_dataset("zeta-alpha-ai/NanoClimateFEVER", name="corpus", split="train")
c = c.to_pandas()

none_count = 0
not_str = 0

for row in c.itertuples():
    if row.text is None:
        none_count += 1
    if not isinstance(row.text, str):
        not_str += 1

print(f"Number of None values in 'text' column: {none_count}")
print(f"Number of non-string values in 'text' column: {not_str}")

Number of None values in 'text' column: 0
Number of non-string values in 'text' column: 0


In [4]:
from datasets import load_dataset

c = load_dataset("nasa-impact/nasa-smd-IR-benchmark", data_files="corpus.jsonl", split="train")
q = load_dataset("nasa-impact/nasa-smd-IR-benchmark", data_files="queries.jsonl", split="train")
qrels = load_dataset("nasa-impact/nasa-smd-IR-benchmark", split="test")

In [5]:
c, q, qrels

(Dataset({
     features: ['_id', 'title', 'text'],
     num_rows: 270206
 }),
 Dataset({
     features: ['_id', 'text'],
     num_rows: 498
 }),
 Dataset({
     features: ['query-id', 'corpus-id', 'score'],
     num_rows: 398
 }))

In [6]:
import pandas as pd

c1 = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/eval/openai_emb_cache/text-embedding-3-large/nasa-smd-IR-benchmark/corpus_embeddings.parquet")
q1 = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/eval/openai_emb_cache/text-embedding-3-large/nasa-smd-IR-benchmark/queries_embeddings.parquet")

c1.shape, q1.shape

((270206, 3), (498, 3))

In [10]:
c1

,id,embeddings,total_tokens
0,0,"[0.022709082812070847, -0.05611325800418854, -...",45133567
1,1,"[-0.007838043384253979, 0.024015283212065697, ...",45133567
2,2,"[0.016265777871012688, 0.006757024209946394, -...",45133567
3,3,"[0.009861164726316929, -0.031135570257902145, ...",45133567
4,4,"[-0.030548231676220894, -0.044901516288518906,...",45133567
...,...,...,...
270201,270201,"[0.016576070338487625, -0.03895174339413643, -...",45133567
270202,270202,"[0.016851386055350304, -0.023938549682497978, ...",45133567
270203,270203,"[0.013844473287463188, -0.003105318872258067, ...",45133567
270204,270204,"[-0.017668748274445534, -0.03445633873343468, ...",45133567


In [11]:
import torch

q2 = torch.Tensor(q1["embeddings"].values.tolist())

In [12]:
len(q2)

498